# 🧠 TRM: Tiny Recursive Model - Sudoku 7M Training

**Production Training on Full Dataset**

---

## 🎯 Objective
Train TRM on **7 million Sudoku puzzles** to achieve **87% test accuracy** (paper benchmark)

## 📊 Paper Specifications
- **Model**: 2-layer Transformer, 512 dim, ~5M parameters
- **Training**: N_sup=16 deep supervision, batch=768 (with gradient accumulation)
- **Iterations**: 60,000 (≈12-24 hours on A100)
- **Optimizer**: AdamW (lr=1e-4, weight_decay=1.0, β=(0.9, 0.95))
- **Stability**: EMA (β=0.999), gradient clipping, stable-max loss

## 🚀 GPU Requirements
- **Recommended**: A100 (40GB) or H100 (80GB)
- **Minimum**: A100 (40GB) - uses gradient accumulation for batch_size=768
- **Runtime**: ~12-24 hours for full training

---

**Reference**: _Less is More: Recursive Reasoning with Tiny Networks_ (arXiv:2510.04871)

## 1️⃣ Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n{'='*60}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {gpu_memory:.1f} GB")
    print(f"\n✓ Ready for training" if gpu_memory >= 40 else f"⚠ Warning: Less than 40GB VRAM")
else:
    print("❌ No GPU detected! Training will be extremely slow.")
print(f"{'='*60}")

In [ ]:
# Install dependencies
!pip install -q datasets wandb tqdm huggingface_hub

print("✓ Dependencies installed")

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, IterableDataset
from datasets import load_dataset
import numpy as np
import time
import json
from tqdm.auto import tqdm
from pathlib import Path

# WandB (optional but recommended)
try:
    import wandb
    WANDB_AVAILABLE = True
except:
    WANDB_AVAILABLE = False
    print("⚠ WandB not available - will skip logging")

print("✓ Imports complete")

## 2️⃣ Configuration (Paper Specifications)

In [ ]:
# Training Configuration - Matches Paper Specifications
CONFIG = {
    # Model Architecture
    'dim': 512,              # Hidden dimension
    'n_layers': 2,           # Transformer layers (kept small)
    'n_heads': 8,            # Attention heads
    'n_latent': 6,           # Latent z-updates per cycle (n in paper)
    'T_cycles': 3,           # Recursion cycles (T in paper)
    'vocab_size': 10,        # 0-9 for Sudoku
    
    # Training Parameters (Paper Spec)
    'batch_size': 48,        # Per-GPU batch (will accumulate to 768)
    'gradient_accumulation': 16,  # 48 × 16 = 768 effective batch
    'max_iters': 60000,      # 60K iterations (paper)
    'warmup_iters': 2000,    # Learning rate warmup
    
    # Optimizer (Paper Spec)
    'lr': 1e-4,              # Learning rate
    'weight_decay': 1.0,     # Strong L2 regularization for Sudoku
    'beta1': 0.9,            # AdamW beta1
    'beta2': 0.95,           # AdamW beta2
    
    # Deep Supervision (Critical!)
    'N_sup': 16,             # Full deep supervision
    'halt_threshold': 0.95,  # ACT early stopping
    'ema_beta': 0.999,       # Exponential Moving Average
    'grad_clip': 1.0,        # Gradient clipping
    
    # Dataset
    'dataset_name': 'sadimanna/1m-sudoku',  # 7M examples
    'streaming': True,       # Memory-efficient loading
    'max_examples': 7000000, # Full dataset
    'num_workers': 4,        # DataLoader workers
    
    # Logging & Checkpointing
    'log_interval': 100,
    'save_interval': 5000,
    'val_interval': 1000,
    'use_wandb': WANDB_AVAILABLE,
    'project_name': 'trm-sudoku-7m',
}

# Calculate effective depth (paper metric)
effective_depth = CONFIG['T_cycles'] * (CONFIG['n_latent'] + 1) * CONFIG['n_layers']

print("="*60)
print("TRM Configuration (Paper Specifications)")
print("="*60)
print(f"Model: {CONFIG['dim']}d, {CONFIG['n_layers']} layers, {CONFIG['n_heads']} heads")
print(f"Recursion: T={CONFIG['T_cycles']} cycles, n={CONFIG['n_latent']} latent updates")
print(f"Effective Depth: {effective_depth} layers")
print(f"Batch Size: {CONFIG['batch_size']} × {CONFIG['gradient_accumulation']} = {CONFIG['batch_size'] * CONFIG['gradient_accumulation']}")
print(f"Deep Supervision: N_sup={CONFIG['N_sup']}")
print(f"Training: {CONFIG['max_iters']:,} iterations")
print(f"Dataset: {CONFIG['max_examples']:,} Sudoku puzzles")
print("="*60)

## 3️⃣ Dataset Loading (7M Examples)

In [ ]:
class StreamingSudokuDataset(IterableDataset):
    """Memory-efficient streaming dataset for 7M Sudoku puzzles."""
    
    def __init__(self, dataset_name, split='train', max_examples=None):
        super().__init__()
        print(f"Loading dataset: {dataset_name} ({split})...")
        self.dataset = load_dataset(dataset_name, split=split, streaming=True)
        self.max_examples = max_examples
    
    def parse_sudoku(self, puzzle_str, solution_str):
        """Convert string format to tensors."""
        try:
            puzzle = torch.tensor([int(c) for c in str(puzzle_str)[:81]], dtype=torch.long)
            solution = torch.tensor([int(c) for c in str(solution_str)[:81]], dtype=torch.long)
            
            # Validate
            if len(puzzle) == 81 and len(solution) == 81:
                return puzzle, solution
        except:
            pass
        return None, None
    
    def __iter__(self):
        count = 0
        for example in self.dataset:
            if self.max_examples and count >= self.max_examples:
                break
            
            # Handle different column names
            if 'puzzle' in example:
                puzzle, solution = self.parse_sudoku(example['puzzle'], example['solution'])
            elif 'quizzes' in example:
                puzzle, solution = self.parse_sudoku(example['quizzes'], example['solutions'])
            else:
                continue
            
            if puzzle is not None:
                # Random initial guess for blanks (1-9)
                y_init = puzzle.clone()
                blank_mask = puzzle == 0
                if blank_mask.any():
                    y_init[blank_mask] = torch.randint(1, 10, (blank_mask.sum(),))
                
                yield {
                    'puzzle': puzzle,
                    'y_init': y_init,
                    'solution': solution
                }
                count += 1

# Load dataset
print("\nLoading Sudoku dataset (streaming mode)...")
train_dataset = StreamingSudokuDataset(
    CONFIG['dataset_name'],
    split='train',
    max_examples=CONFIG['max_examples']
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    pin_memory=True
)

# Test loading
print("Testing dataset loading...")
test_batch = next(iter(train_loader))
print(f"✓ Batch loaded: {test_batch['puzzle'].shape}")
print(f"  Puzzle range: {test_batch['puzzle'].min()}-{test_batch['puzzle'].max()}")
print(f"  Solution range: {test_batch['solution'].min()}-{test_batch['solution'].max()}")
print(f"  Blanks per puzzle: {(test_batch['puzzle'] == 0).float().sum(dim=1).mean():.1f}")
print("✓ Dataset ready")

## 4️⃣ Model Architecture (TRM)

In [ ]:
# Model Components

class RMSNorm(nn.Module):
    """Root Mean Square Normalization (paper spec)."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class SwiGLU(nn.Module):
    """SwiGLU activation (paper spec)."""
    def __init__(self, in_dim):
        super().__init__()
        # Hidden dimension (8/3 expansion)
        hidden = ((int(2/3 * 4 * in_dim) + 63) // 64) * 64
        self.w1 = nn.Linear(in_dim, hidden, bias=False)
        self.w2 = nn.Linear(hidden, in_dim, bias=False)
        self.w3 = nn.Linear(in_dim, hidden, bias=False)
    
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class TinyBlock(nn.Module):
    """Single transformer block."""
    def __init__(self, dim, n_heads=8):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True, bias=False)
        self.norm2 = RMSNorm(dim)
        self.ffn = SwiGLU(dim)
    
    def forward(self, x):
        # Pre-norm architecture
        h = self.norm1(x)
        x = x + self.attn(h, h, h, need_weights=False)[0]
        x = x + self.ffn(self.norm2(x))
        return x


class TinyNet(nn.Module):
    """Tiny network (shared for all recursions)."""
    def __init__(self, dim=512, n_layers=2, n_heads=8):
        super().__init__()
        self.blocks = nn.ModuleList([TinyBlock(dim, n_heads) for _ in range(n_layers)])
        self.norm = RMSNorm(dim)
    
    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.norm(x)

print("✓ Model components defined")

In [ ]:
class TRMModel(nn.Module):
    """Tiny Recursive Model - Algorithm 3 from paper."""
    
    def __init__(self, dim=512, n_layers=2, n_heads=8, n_latent=6, T_cycles=3, vocab_size=10):
        super().__init__()
        self.dim = dim
        self.n_latent = n_latent  # n in paper
        self.T_cycles = T_cycles   # T in paper
        
        # Single tiny network (shared)
        self.net = TinyNet(dim, n_layers, n_heads)
        
        # Projections
        self.z_proj = nn.Linear(3 * dim, dim, bias=False)  # Input: [x, y, z]
        self.y_proj = nn.Linear(2 * dim, dim, bias=False)  # Input: [y, z]
        
        # Heads
        self.output_head = nn.Linear(dim, vocab_size, bias=False)
        self.q_head = nn.Linear(dim, 1, bias=False)  # ACT halting head
    
    def latent_recursion(self, x, y, z):
        """Single latent recursion cycle: n z-updates + 1 y-update."""
        # n latent z-updates
        for _ in range(self.n_latent):
            z_input = torch.cat([x, y, z], dim=-1)
            z = self.net(self.z_proj(z_input))
        
        # 1 y-refine
        y_input = torch.cat([y, z], dim=-1)
        y = self.net(self.y_proj(y_input))
        
        return y, z
    
    def forward(self, x, y, z):
        """Deep recursion: T-1 no-grad warmups + 1 grad cycle."""
        # T-1 cycles without gradients (warmup)
        with torch.no_grad():
            for _ in range(self.T_cycles - 1):
                y, z = self.latent_recursion(x, y, z)
        
        # Final cycle with gradients
        y, z = self.latent_recursion(x, y, z)
        
        # Output heads
        y_hat = self.output_head(y)  # [B, L, vocab]
        q_hat = self.q_head(y.mean(dim=1))  # [B, 1] halting logit
        
        return (y, z), y_hat, q_hat
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Initialize model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = TRMModel(
    dim=CONFIG['dim'],
    n_layers=CONFIG['n_layers'],
    n_heads=CONFIG['n_heads'],
    n_latent=CONFIG['n_latent'],
    T_cycles=CONFIG['T_cycles'],
    vocab_size=CONFIG['vocab_size']
).to(device)

# Token embedding (separate from model)
embedding = nn.Embedding(CONFIG['vocab_size'], CONFIG['dim']).to(device)

print("="*60)
print("TRM Model Initialized")
print("="*60)
print(f"Total Parameters: {model.count_parameters():,}")
print(f"Embedding Parameters: {sum(p.numel() for p in embedding.parameters()):,}")
print(f"Total (Model + Embedding): {model.count_parameters() + sum(p.numel() for p in embedding.parameters()):,}")
print(f"Device: {device}")
print("="*60)

## 5️⃣ Training Loop (Deep Supervision)

In [ ]:
# Optimizer setup (paper spec)
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(embedding.parameters()),
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
    betas=(CONFIG['beta1'], CONFIG['beta2'])
)

# Mixed precision scaler (for A100/H100)
scaler = torch.amp.GradScaler('cuda') if device == 'cuda' else None

# EMA for stable generalization
ema_params = {}
for name, p in model.named_parameters():
    if p.requires_grad:
        ema_params[f'model_{name}'] = p.data.clone()
for name, p in embedding.named_parameters():
    if p.requires_grad:
        ema_params[f'emb_{name}'] = p.data.clone()

def update_ema():
    """Update EMA parameters."""
    beta = CONFIG['ema_beta']
    for name, p in model.named_parameters():
        if p.requires_grad:
            key = f'model_{name}'
            ema_params[key] = beta * ema_params[key] + (1 - beta) * p.data
    for name, p in embedding.named_parameters():
        if p.requires_grad:
            key = f'emb_{name}'
            ema_params[key] = beta * ema_params[key] + (1 - beta) * p.data

print("✓ Optimizer and EMA initialized")

In [ ]:
# Initialize WandB (optional)
if CONFIG['use_wandb']:
    wandb.init(
        project=CONFIG['project_name'],
        config=CONFIG,
        name=f"trm_sudoku_{CONFIG['max_iters']}iters"
    )
    print("✓ WandB initialized")
else:
    print("⚠ WandB disabled - using local logging only")

In [ ]:
# Main Training Loop
print("\n" + "="*60)
print("Starting Training")
print("="*60)

model.train()
train_iter = iter(train_loader)
start_time = time.time()
best_acc = 0.0
global_step = 0

# Training statistics
running_loss = 0.0
running_acc = 0.0
running_sup_steps = 0.0

progress_bar = tqdm(range(CONFIG['max_iters']), desc="Training")

for step in progress_bar:
    global_step = step
    
    # Learning rate warmup
    if step < CONFIG['warmup_iters']:
        lr = CONFIG['lr'] * (step + 1) / CONFIG['warmup_iters']
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
    
    # Get batch
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        batch = next(train_iter)
    
    puzzle = batch['puzzle'].to(device)
    y_init_tokens = batch['y_init'].to(device)
    solution = batch['solution'].to(device)
    
    # Initial embeddings
    x = embedding(puzzle)
    y = embedding(y_init_tokens)
    z = torch.zeros_like(x)
    
    total_loss = 0.0
    sup_steps_taken = 0
    
    # Deep Supervision Loop (N_sup=16)
    for sup_step in range(CONFIG['N_sup']):
        with torch.amp.autocast('cuda', enabled=(device=='cuda')):
            # Forward pass
            (y_new, z_new), y_hat, q_hat = model(x, y, z)
            
            # Cross-entropy loss
            ce_loss = F.cross_entropy(
                y_hat.view(-1, CONFIG['vocab_size']),
                solution.view(-1),
                ignore_index=0  # Ignore padding if any
            )
            
            # ACT halting loss
            with torch.no_grad():
                preds = y_hat.argmax(-1)
                acc_per_sample = (preds == solution).float().mean(dim=-1)
                halt_target = (acc_per_sample > 0.99).float().unsqueeze(-1)
            
            act_loss = F.binary_cross_entropy_with_logits(q_hat, halt_target)
            loss = ce_loss + 0.5 * act_loss
        
        # Backprop and optimize (within deep supervision loop)
        loss_scaled = loss / CONFIG['gradient_accumulation']
        if scaler:
            scaler.scale(loss_scaled).backward()
        else:
            loss_scaled.backward()
        
        # Gradient accumulation
        if (sup_step + 1) % CONFIG['gradient_accumulation'] == 0 or sup_step == CONFIG['N_sup'] - 1:
            if scaler:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
                optimizer.step()
            
            optimizer.zero_grad()
            update_ema()
        
        total_loss += loss.item()
        sup_steps_taken += 1
        
        # Early halt if confident
        halt_prob = torch.sigmoid(q_hat).mean().item()
        if halt_prob > CONFIG['halt_threshold']:
            break
        
        # Detach for next supervision step
        y = y_new.detach()
        z = z_new.detach()
    
    # Statistics
    with torch.no_grad():
        train_acc = (y_hat.argmax(-1) == solution).float().mean().item()
    
    running_loss += total_loss / sup_steps_taken
    running_acc += train_acc
    running_sup_steps += sup_steps_taken
    
    # Logging
    if (step + 1) % CONFIG['log_interval'] == 0:
        avg_loss = running_loss / CONFIG['log_interval']
        avg_acc = running_acc / CONFIG['log_interval']
        avg_sup = running_sup_steps / CONFIG['log_interval']
        elapsed = time.time() - start_time
        
        progress_bar.set_postfix({
            'loss': f"{avg_loss:.4f}",
            'acc': f"{avg_acc:.3f}",
            'sup': f"{avg_sup:.1f}"
        })
        
        if CONFIG['use_wandb']:
            wandb.log({
                'train/loss': avg_loss,
                'train/accuracy': avg_acc,
                'train/sup_steps': avg_sup,
                'train/halt_prob': halt_prob,
                'time_elapsed': elapsed,
                'step': step
            })
        
        running_loss = 0.0
        running_acc = 0.0
        running_sup_steps = 0.0
    
    # Checkpointing
    if (step + 1) % CONFIG['save_interval'] == 0:
        checkpoint = {
            'step': step,
            'model': model.state_dict(),
            'embedding': embedding.state_dict(),
            'optimizer': optimizer.state_dict(),
            'ema_params': ema_params,
            'config': CONFIG,
            'best_acc': best_acc
        }
        torch.save(checkpoint, f'/content/trm_checkpoint_{step+1}.pt')
        print(f"\n✓ Checkpoint saved at step {step+1}")

# Final save
final_checkpoint = {
    'step': CONFIG['max_iters'],
    'model': model.state_dict(),
    'embedding': embedding.state_dict(),
    'optimizer': optimizer.state_dict(),
    'ema_params': ema_params,
    'config': CONFIG,
}
torch.save(final_checkpoint, '/content/trm_final.pt')

print("\n" + "="*60)
print("✓ Training Complete!")
print(f"Total time: {(time.time() - start_time) / 3600:.2f} hours")
print("="*60)

if CONFIG['use_wandb']:
    wandb.finish()

## 6️⃣ Evaluation with EMA Weights

In [ ]:
# Load best checkpoint and apply EMA weights
checkpoint = torch.load('/content/trm_final.pt', map_location=device)
model.load_state_dict(checkpoint['model'])
embedding.load_state_dict(checkpoint['embedding'])

# Apply EMA weights for evaluation
for name, p in model.named_parameters():
    key = f'model_{name}'
    if key in checkpoint['ema_params']:
        p.data = checkpoint['ema_params'][key]

for name, p in embedding.named_parameters():
    key = f'emb_{name}'
    if key in checkpoint['ema_params']:
        p.data = checkpoint['ema_params'][key]

model.eval()
print("✓ Model loaded with EMA weights")

In [ ]:
# Evaluation on test set
print("\nEvaluating on test set...")

test_dataset = StreamingSudokuDataset(
    CONFIG['dataset_name'],
    split='test' if 'test' in CONFIG['dataset_name'] else 'train',
    max_examples=10000  # Evaluate on 10K examples
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    num_workers=2
)

total_correct_cells = 0
total_cells = 0
total_correct_grids = 0
total_grids = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        puzzle = batch['puzzle'].to(device)
        y_init = batch['y_init'].to(device)
        solution = batch['solution'].to(device)
        
        x = embedding(puzzle)
        y = embedding(y_init)
        z = torch.zeros_like(x)
        
        # Run full N_sup iterations at test time
        for _ in range(CONFIG['N_sup']):
            (y, z), y_hat, q_hat = model(x, y, z)
            if torch.sigmoid(q_hat).mean() > CONFIG['halt_threshold']:
                break
        
        preds = y_hat.argmax(-1)
        
        # Cell-level accuracy
        total_correct_cells += (preds == solution).sum().item()
        total_cells += solution.numel()
        
        # Grid-level accuracy (all 81 cells correct)
        total_correct_grids += (preds == solution).all(dim=-1).sum().item()
        total_grids += puzzle.size(0)

cell_acc = 100 * total_correct_cells / total_cells
grid_acc = 100 * total_correct_grids / total_grids

print("\n" + "="*60)
print("Final Evaluation Results")
print("="*60)
print(f"Cell Accuracy: {cell_acc:.2f}%")
print(f"Grid Accuracy: {grid_acc:.2f}% (Target: 87% from paper)")
print(f"Evaluated on: {total_grids:,} puzzles")
print("="*60)

## 7️⃣ Download Checkpoints

In [ ]:
# Download trained model
from google.colab import files

print("Downloading checkpoints...")
files.download('/content/trm_final.pt')

# List all checkpoints
import glob
checkpoints = glob.glob('/content/trm_checkpoint_*.pt')
print(f"\nAvailable checkpoints: {len(checkpoints)}")
for ckpt in checkpoints[-3:]:
    print(f"  {ckpt}")